# Similarity Grad-CAM Explanations

This notebook adds local explanations for the sketch-to-photo retrieval model.

The model is not predicting a class label, so the explanation target is the scalar similarity score between one sketch embedding and one photo embedding. For each selected pair, the notebook backpropagates that similarity into the ResNet-18 `layer4` feature maps and into the input pixels.

Outputs:

- sketch saliency map
- sketch Grad-CAM overlay
- optional photo Grad-CAM overlay
- score and retrieval rank context

The same explanation helpers are used for both CUFSF paired examples and FEI generated-sketch gallery examples.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import random

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'matplotlib'))

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib import cm
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Works whether the kernel starts in the repo root or in notebooks/.
CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / 'notebooks').exists() else CWD.parent
PROJECT_ROOT = REPO_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'

STD_ROOT = DATA_ROOT / 'processed' / 'standardized_224_neutral'
STD_CUFSF_DIR = STD_ROOT / 'cufsf'
STD_FEI_DIR = STD_ROOT / 'fei'

OUTPUT_DIR = DATA_ROOT / 'outputs' / 'multidataset_standardized_resnet18'
MODEL_DIR = OUTPUT_DIR / 'models'
EXPLANATION_DIR = OUTPUT_DIR / 'explanations'
EXPLANATION_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 224
PROJECTION_DIM = 128
BATCH_SIZE = 64
NUM_WORKERS = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PIN_MEMORY = DEVICE == 'cuda'

print('Repo root:', REPO_ROOT)
print('Data root:', DATA_ROOT)
print('Device:', DEVICE)
print('Explanation output dir:', EXPLANATION_DIR)

## 2. Model and preprocessing

This is the same projection encoder architecture used by the training notebooks. The checkpoint is loaded with `pretrained=False` because the learned weights come from the saved contrastive model.

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class ResNet18ProjectionEncoder(nn.Module):
    def __init__(self, projection_dim=128, pretrained=False):
        super().__init__()
        if pretrained:
            try:
                weights = models.ResNet18_Weights.DEFAULT
                backbone = models.resnet18(weights=weights)
                print('Loaded ImageNet pretrained ResNet18 weights.')
            except Exception as exc:
                print('Could not load pretrained weights; using random initialization.')
                print('Reason:', repr(exc))
                backbone = models.resnet18(weights=None)
        else:
            backbone = models.resnet18(weights=None)

        feature_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projection = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(inplace=True),
            nn.Linear(feature_dim, projection_dim),
        )

    def forward(self, x):
        features = self.backbone(x)
        projected = self.projection(features)
        return F.normalize(projected, p=2, dim=1)


def torch_load_checkpoint(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def load_checkpoint(path, device=DEVICE):
    if not path.exists():
        raise FileNotFoundError(
            f'Missing checkpoint: {path}\n'
            'Run notebook 04 once, or update MODEL_DIR to point at the saved model.'
        )

    checkpoint = torch_load_checkpoint(path, device)
    projection_dim = checkpoint.get('config', {}).get('projection_dim', PROJECTION_DIM)
    model = ResNet18ProjectionEncoder(projection_dim=projection_dim, pretrained=False).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    print('Loaded checkpoint:', path)
    print('Checkpoint epoch:', checkpoint.get('epoch'))
    print('Projection dim:', projection_dim)
    return model, checkpoint


BEST_MODEL_PATH = MODEL_DIR / 'best_multidataset_standardized_resnet18.pt'
best_model, best_checkpoint = load_checkpoint(BEST_MODEL_PATH)

## 3. Retrieval table helpers

These helpers rank sketches against photos so the explanations can include the same retrieval context as the earlier notebooks.

In [ ]:
class ImageTableDataset(Dataset):
    def __init__(self, df, path_column, transform):
        self.df = df.reset_index(drop=True).copy()
        self.path_column = path_column
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row[self.path_column]).convert('RGB')
        return {
            'id': str(row['id']),
            'image': self.transform(image),
            'path': row[self.path_column],
        }


@torch.no_grad()
def encode_image_table(model, df, path_column, label, transform=eval_transform, device=DEVICE):
    dataset = ImageTableDataset(df, path_column, transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    ids = []
    embeddings = []
    model.eval()
    for batch in tqdm(loader, desc=label, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        embeddings.append(model(images).detach().cpu())
        ids.extend([str(x) for x in batch['id']])
    return ids, torch.cat(embeddings, dim=0).numpy()


def retrieval_metrics_from_ranks(ranks, ks=(1, 5, 10)):
    ranks = np.asarray(ranks)
    metrics = {f'Recall@{k}': float(np.mean(ranks <= k)) for k in ks}
    metrics['MRR'] = float(np.mean(1.0 / ranks))
    metrics['Mean Rank'] = float(np.mean(ranks))
    metrics['Median Rank'] = float(np.median(ranks))
    return metrics


def rank_paired_table(model, df, label):
    df = df.reset_index(drop=True).copy()
    df['id'] = df['id'].astype(str)
    sketch_ids, sketch_embeddings = encode_image_table(model, df, 'sketch_path', f'{label}: sketches')
    photo_ids, photo_embeddings = encode_image_table(model, df, 'photo_path', f'{label}: photos')
    similarity = sketch_embeddings @ photo_embeddings.T

    photo_id_to_index = {gid: idx for idx, gid in enumerate(photo_ids)}
    rows = []
    ranks = []
    for query_idx, query_id in enumerate(sketch_ids):
        correct_idx = photo_id_to_index[query_id]
        ranked_indices = np.argsort(-similarity[query_idx])
        rank = int(np.where(ranked_indices == correct_idx)[0][0]) + 1
        top_idx = int(ranked_indices[0])
        ranks.append(rank)
        rows.append({
            'row_index': query_idx,
            'id': query_id,
            'rank': rank,
            'top_id': photo_ids[top_idx],
            'top_index': top_idx,
            'correct_index': correct_idx,
            'top_score': float(similarity[query_idx, top_idx]),
            'correct_score': float(similarity[query_idx, correct_idx]),
            'sketch_path': df.iloc[query_idx]['sketch_path'],
            'photo_path': df.iloc[correct_idx]['photo_path'],
            'correct_photo_path': df.iloc[correct_idx]['photo_path'],
            'top_photo_path': df.iloc[top_idx]['photo_path'],
        })

    ranked_df = pd.DataFrame(rows)
    metrics = retrieval_metrics_from_ranks(ranks)
    return metrics, np.asarray(ranks), similarity, ranked_df


def rank_query_gallery(model, query_df, gallery_df, label):
    query_df = query_df.reset_index(drop=True).copy()
    gallery_df = gallery_df.reset_index(drop=True).copy()
    query_df['id'] = query_df['id'].astype(str)
    gallery_df['id'] = gallery_df['id'].astype(str)

    query_ids, query_embeddings = encode_image_table(model, query_df, 'sketch_path', f'{label}: queries')
    gallery_ids, gallery_embeddings = encode_image_table(model, gallery_df, 'photo_path', f'{label}: gallery')
    similarity = query_embeddings @ gallery_embeddings.T

    gallery_id_to_index = {gid: idx for idx, gid in enumerate(gallery_ids)}
    rows = []
    ranks = []
    for query_idx, query_id in enumerate(query_ids):
        correct_idx = gallery_id_to_index[query_id]
        ranked_indices = np.argsort(-similarity[query_idx])
        rank = int(np.where(ranked_indices == correct_idx)[0][0]) + 1
        top_idx = int(ranked_indices[0])
        ranks.append(rank)
        rows.append({
            'query_index': query_idx,
            'id': query_id,
            'rank': rank,
            'top_id': gallery_ids[top_idx],
            'top_index': top_idx,
            'correct_index': correct_idx,
            'top_score': float(similarity[query_idx, top_idx]),
            'correct_score': float(similarity[query_idx, correct_idx]),
            'sketch_path': query_df.iloc[query_idx]['sketch_path'],
            'correct_photo_path': gallery_df.iloc[correct_idx]['photo_path'],
            'top_photo_path': gallery_df.iloc[top_idx]['photo_path'],
        })

    ranked_df = pd.DataFrame(rows)
    metrics = retrieval_metrics_from_ranks(ranks)
    return metrics, np.asarray(ranks), similarity, ranked_df


def choose_representative_rows(ranked_df, n=3):
    ranked_df = ranked_df.reset_index(drop=True)
    candidates = []
    candidates.append(ranked_df.sort_values('rank', ascending=True).head(1))

    median_rank = ranked_df['rank'].median()
    median_like = ranked_df.iloc[(ranked_df['rank'] - median_rank).abs().argsort()[:1]]
    candidates.append(median_like)

    if (ranked_df['rank'] > 1).any():
        candidates.append(ranked_df.sort_values('rank', ascending=False).head(1))

    selected = pd.concat(candidates, ignore_index=True).drop_duplicates('id').head(n)
    if len(selected) < n:
        filler = ranked_df.sample(min(n - len(selected), len(ranked_df)), random_state=SEED)
        selected = pd.concat([selected, filler], ignore_index=True).drop_duplicates('id').head(n)
    return selected.reset_index(drop=True)


def print_metrics(label, metrics):
    print(label)
    print('-' * len(label))
    for key, value in metrics.items():
        if key.startswith('Recall') or key == 'MRR':
            print(f'{key}: {value:.4f}')
        else:
            print(f'{key}: {value:.2f}')

## 4. Similarity saliency and Grad-CAM helpers

`compute_similarity_explanations` is the core idea:

1. Encode the reference image without gradients.
2. Encode the image being explained with gradients enabled.
3. Use cosine similarity between the two embeddings as the scalar target.
4. Backpropagate that scalar into `model.backbone.layer4` and the normalized input tensor.
5. Convert the gradients into a saliency map and a Grad-CAM heatmap.

Because the same encoder processes sketches and photos, the helper computes the sketch explanation first and can compute the photo explanation separately with the sketch embedding detached as the reference.

In [ ]:
def load_rgb_image(path, size=IMAGE_SIZE):
    return Image.open(path).convert('RGB').resize((size, size), Image.Resampling.BICUBIC)


def tensor_from_image(image):
    return eval_transform(image).unsqueeze(0)


def normalize_map(values, eps=1e-8):
    values = np.asarray(values, dtype=np.float32)
    values = values - float(values.min())
    denom = float(values.max())
    if denom < eps:
        return np.zeros_like(values, dtype=np.float32)
    return values / denom


def gradcam_from_tensors(activations, gradients, output_size=(IMAGE_SIZE, IMAGE_SIZE)):
    weights = gradients.mean(dim=(2, 3), keepdim=True)
    cam_tensor = (weights * activations).sum(dim=1, keepdim=True)
    cam_tensor = F.relu(cam_tensor)
    cam_tensor = F.interpolate(cam_tensor, size=output_size, mode='bilinear', align_corners=False)
    return normalize_map(cam_tensor[0, 0].detach().cpu().numpy())


class Layer4Capture:
    def __init__(self, layer):
        self.activations = None
        self.gradients = None
        self.handle = layer.register_forward_hook(self.forward_hook)

    def forward_hook(self, module, inputs, output):
        self.activations = output
        output.register_hook(self.save_gradient)

    def save_gradient(self, gradient):
        self.gradients = gradient

    def close(self):
        self.handle.remove()


def compute_branch_explanation(model, explained_tensor, reference_tensor, target_layer=None, device=DEVICE):
    target_layer = target_layer or model.backbone.layer4
    model.eval()

    reference_tensor = reference_tensor.to(device)
    explained_tensor = explained_tensor.detach().clone().to(device).requires_grad_(True)

    with torch.no_grad():
        reference_embedding = model(reference_tensor)

    model.zero_grad(set_to_none=True)
    capture = Layer4Capture(target_layer)
    try:
        explained_embedding = model(explained_tensor)
        similarity_score = F.cosine_similarity(explained_embedding, reference_embedding, dim=1).mean()
        similarity_score.backward()

        if capture.activations is None or capture.gradients is None:
            raise RuntimeError('Layer4 activations or gradients were not captured.')

        saliency = explained_tensor.grad.detach().abs().max(dim=1)[0][0].cpu().numpy()
        saliency = normalize_map(saliency)
        gradcam = gradcam_from_tensors(capture.activations, capture.gradients)

        return {
            'score': float(similarity_score.detach().cpu()),
            'saliency': saliency,
            'gradcam': gradcam,
        }
    finally:
        capture.close()
        model.zero_grad(set_to_none=True)


def compute_similarity_explanations(model, sketch_path, photo_path, explain_photo=True, device=DEVICE):
    sketch_image = load_rgb_image(sketch_path)
    photo_image = load_rgb_image(photo_path)
    sketch_tensor = tensor_from_image(sketch_image)
    photo_tensor = tensor_from_image(photo_image)

    sketch_explanation = compute_branch_explanation(
        model,
        explained_tensor=sketch_tensor,
        reference_tensor=photo_tensor,
        device=device,
    )

    photo_explanation = None
    if explain_photo:
        photo_explanation = compute_branch_explanation(
            model,
            explained_tensor=photo_tensor,
            reference_tensor=sketch_tensor,
            device=device,
        )

    return {
        'score': sketch_explanation['score'],
        'sketch_path': str(sketch_path),
        'photo_path': str(photo_path),
        'sketch_image': sketch_image,
        'photo_image': photo_image,
        'sketch_saliency': sketch_explanation['saliency'],
        'sketch_gradcam': sketch_explanation['gradcam'],
        'photo_gradcam': None if photo_explanation is None else photo_explanation['gradcam'],
    }


def overlay_heatmap(image, heatmap, alpha=0.48, cmap_name='jet'):
    base = np.asarray(image.convert('L'), dtype=np.float32) / 255.0
    base_rgb = np.repeat(base[..., None], 3, axis=2)
    color = cm.get_cmap(cmap_name)(normalize_map(heatmap))[..., :3]
    overlay = (1 - alpha) * base_rgb + alpha * color
    return np.clip(overlay, 0, 1)


def make_explanation_figure(explanation, title, save_path=None, show_photo_gradcam=True):
    columns = 5 if show_photo_gradcam and explanation.get('photo_gradcam') is not None else 4
    fig, axes = plt.subplots(1, columns, figsize=(3.2 * columns, 3.6))
    if columns == 1:
        axes = [axes]

    axes[0].imshow(explanation['sketch_image'].convert('L'), cmap='gray')
    axes[0].set_title('Query sketch')
    axes[0].axis('off')

    axes[1].imshow(explanation['photo_image'].convert('L'), cmap='gray')
    axes[1].set_title('Target photo')
    axes[1].axis('off')

    axes[2].imshow(overlay_heatmap(explanation['sketch_image'], explanation['sketch_saliency'], alpha=0.55, cmap_name='magma'))
    axes[2].set_title('Sketch saliency')
    axes[2].axis('off')

    axes[3].imshow(overlay_heatmap(explanation['sketch_image'], explanation['sketch_gradcam']))
    axes[3].set_title('Sketch Grad-CAM')
    axes[3].axis('off')

    if columns == 5:
        axes[4].imshow(overlay_heatmap(explanation['photo_image'], explanation['photo_gradcam']))
        axes[4].set_title('Photo Grad-CAM')
        axes[4].axis('off')

    fig.suptitle(f"{title}\ncosine similarity = {explanation['score']:.3f}", fontsize=13)
    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches='tight')
        print('Saved:', save_path)
    if 'agg' not in plt.get_backend().lower():
        plt.show()
    return fig


def build_top_vs_correct_explanations(model, row, device=DEVICE):
    sketch_path = row['sketch_path']
    correct_photo_path = row.get('correct_photo_path', row.get('photo_path'))
    top_photo_path = row.get('top_photo_path', correct_photo_path)
    top_id = str(row.get('top_id', row['id']))
    correct_id = str(row['id'])
    rank = int(row['rank'])

    top_explanation = compute_similarity_explanations(
        model,
        sketch_path=sketch_path,
        photo_path=top_photo_path,
        explain_photo=True,
        device=device,
    )
    top_explanation.update({
        'target_kind': 'top target',
        'target_id': top_id,
        'target_rank': 1,
        'retrieval_score': float(row['top_score']),
    })

    if Path(top_photo_path) == Path(correct_photo_path):
        correct_explanation = dict(top_explanation)
    else:
        correct_explanation = compute_similarity_explanations(
            model,
            sketch_path=sketch_path,
            photo_path=correct_photo_path,
            explain_photo=True,
            device=device,
        )
    correct_explanation.update({
        'target_kind': 'correct target',
        'target_id': correct_id,
        'target_rank': rank,
        'retrieval_score': float(row['correct_score']),
    })
    return [top_explanation, correct_explanation]


def make_top_vs_correct_explanation_figure(explanations, title, save_path=None):
    fig, axes = plt.subplots(2, 4, figsize=(13.5, 7.0))
    for row_index, explanation in enumerate(explanations):
        axes[row_index, 0].imshow(explanation['sketch_image'].convert('L'), cmap='gray')
        axes[row_index, 0].set_title('Query sketch')
        axes[row_index, 0].axis('off')

        target_title = (
            f"{explanation['target_kind']} {explanation['target_id']}\n"
            f"rank {explanation['target_rank']} | score {explanation['retrieval_score']:.3f}"
        )
        axes[row_index, 1].imshow(explanation['photo_image'].convert('L'), cmap='gray')
        axes[row_index, 1].set_title(target_title)
        axes[row_index, 1].axis('off')

        axes[row_index, 2].imshow(overlay_heatmap(explanation['sketch_image'], explanation['sketch_gradcam']))
        axes[row_index, 2].set_title('Sketch Grad-CAM')
        axes[row_index, 2].axis('off')

        axes[row_index, 3].imshow(overlay_heatmap(explanation['photo_image'], explanation['photo_gradcam']))
        axes[row_index, 3].set_title('Photo Grad-CAM')
        axes[row_index, 3].axis('off')

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=170, bbox_inches='tight')
        print('Saved:', save_path)
    if 'agg' not in plt.get_backend().lower():
        plt.show()
    return fig

## 5. CUFSF paired examples

This section explains CUFSF retrieval decisions by comparing the model's rank-1 photo with the actual correct photo. For rank-1 successes, the two rows will show the same target. For misses, the contrast makes it easier to see which evidence drove the wrong top retrieval versus the true match.

In [ ]:
def load_cufsf_test_pairs():
    candidates = [
        STD_CUFSF_DIR / 'test_pairs.csv',
        STD_CUFSF_DIR / 'test' / 'pairs.csv',
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_csv(path, dtype={'id': str})
            df['id'] = df['id'].astype(str)
            print('Loaded CUFSF test pairs:', path, 'rows:', len(df))
            return df
    raise FileNotFoundError(f'Could not find CUFSF test pairs in: {candidates}')


cufsf_test_df = load_cufsf_test_pairs()
cufsf_metrics, cufsf_ranks, cufsf_similarity, cufsf_ranked_df = rank_paired_table(
    best_model,
    cufsf_test_df,
    label='CUFSF test',
)
print_metrics('CUFSF paired retrieval', cufsf_metrics)
display(cufsf_ranked_df.sort_values('rank').head(8))

In [ ]:
CUFSF_EXAMPLES = 3
cufsf_examples = choose_representative_rows(cufsf_ranked_df, n=CUFSF_EXAMPLES)
display(cufsf_examples[['id', 'rank', 'top_id', 'correct_score', 'top_score']])

print('CUFSF top-vs-correct explanations')
for example_number, row in cufsf_examples.iterrows():
    explanations = build_top_vs_correct_explanations(best_model, row)
    title = f"CUFSF id {row['id']} | top target {row['top_id']} vs correct target {row['id']} | correct rank {int(row['rank'])}"
    save_path = EXPLANATION_DIR / f"cufsf_top_vs_correct_gradcam_{example_number + 1}_{row['id']}.png"
    make_top_vs_correct_explanation_figure(explanations, title=title, save_path=save_path)

## 6. FEI generated-sketch examples

This section explains retrieval decisions for generated FEI sketch queries against a FEI photo gallery. By default it uses the controlled orientation-13 gallery because notebook 04 found it was the stronger FEI setting.

Each example shows the model's rank-1 photo and the actual correct photo. This is especially useful for FEI misses, where the wrong top target and the true target often reveal different attention patterns.

In [ ]:
FEI_GALLERY_NAME = 'controlled_orientation_13_gallery'


def load_fei_bundle(gallery_name=FEI_GALLERY_NAME):
    bundle_dir = STD_FEI_DIR / gallery_name
    query_csv = bundle_dir / 'query_sketches.csv'
    gallery_csv = bundle_dir / 'gallery_photos.csv'
    if not query_csv.exists() or not gallery_csv.exists():
        available = sorted(p.name for p in STD_FEI_DIR.glob('*') if p.is_dir()) if STD_FEI_DIR.exists() else []
        raise FileNotFoundError(
            f'Missing FEI processed tables for {gallery_name}.\n'
            f'Expected: {query_csv} and {gallery_csv}\n'
            f'Available FEI processed folders: {available}\n'
            'Run the FEI table-building cells in notebook 04 if needed.'
        )
    query_df = pd.read_csv(query_csv, dtype={'id': str, 'raw_id': str})
    gallery_df = pd.read_csv(gallery_csv, dtype={'id': str, 'raw_id': str})
    print('Loaded FEI queries:', query_csv, 'rows:', len(query_df))
    print('Loaded FEI gallery:', gallery_csv, 'rows:', len(gallery_df))
    return query_df, gallery_df


fei_query_df, fei_gallery_df = load_fei_bundle(FEI_GALLERY_NAME)
fei_metrics, fei_ranks, fei_similarity, fei_ranked_df = rank_query_gallery(
    best_model,
    fei_query_df,
    fei_gallery_df,
    label=f'FEI {FEI_GALLERY_NAME}',
)
print_metrics(f'FEI {FEI_GALLERY_NAME} retrieval', fei_metrics)
display(fei_ranked_df.sort_values('rank').head(8))

In [ ]:
FEI_EXAMPLES = 3
fei_examples = choose_representative_rows(fei_ranked_df, n=FEI_EXAMPLES)
display(fei_examples[['id', 'rank', 'top_id', 'correct_score', 'top_score']])

print('FEI top-vs-correct explanations')
for example_number, row in fei_examples.iterrows():
    explanations = build_top_vs_correct_explanations(best_model, row)
    title = (
        f"FEI query {row['id']} | top target {row['top_id']} vs correct target {row['id']} | "
        f"correct rank {int(row['rank'])}"
    )
    save_path = EXPLANATION_DIR / f"fei_{FEI_GALLERY_NAME}_top_vs_correct_gradcam_{example_number + 1}_{row['id']}.png"
    make_top_vs_correct_explanation_figure(explanations, title=title, save_path=save_path)

## 7. One-off custom pair

Use this final cell when you want to explain a specific image pair for the presentation. Point `CUSTOM_SKETCH_PATH` and `CUSTOM_PHOTO_PATH` at any standardized sketch/photo pair, then run the cell.

In [ ]:
CUSTOM_SKETCH_PATH = None
CUSTOM_PHOTO_PATH = None

if CUSTOM_SKETCH_PATH and CUSTOM_PHOTO_PATH:
    custom_explanation = compute_similarity_explanations(
        best_model,
        sketch_path=CUSTOM_SKETCH_PATH,
        photo_path=CUSTOM_PHOTO_PATH,
        explain_photo=True,
    )
    make_explanation_figure(
        custom_explanation,
        title='Custom sketch/photo similarity explanation',
        save_path=EXPLANATION_DIR / 'custom_similarity_gradcam.png',
        show_photo_gradcam=True,
    )
else:
    print('Set CUSTOM_SKETCH_PATH and CUSTOM_PHOTO_PATH to explain a custom pair.')